In [30]:
!pip install pyspark

In [31]:
from pyspark.sql import SparkSession

In [32]:
spark = SparkSession.builder.appName('PySpark-ML').getOrCreate()

In [33]:
df=spark.read.csv('/content/Mall_Customers.csv',header=True,inferSchema=True)

In [34]:
df.show()

+----------+------+---+------------------+----------------------+
|CustomerID|Gender|Age|Annual Income (k$)|Spending Score (1-100)|
+----------+------+---+------------------+----------------------+
|         1|  Male| 19|                15|                    39|
|         2|  Male| 21|                15|                    81|
|         3|Female| 20|                16|                     6|
|         4|Female| 23|                16|                    77|
|         5|Female| 31|                17|                    40|
|         6|Female| 22|                17|                    76|
|         7|Female| 35|                18|                     6|
|         8|Female| 23|                18|                    94|
|         9|  Male| 64|                19|                     3|
|        10|Female| 30|                19|                    72|
|        11|  Male| 67|                19|                    14|
|        12|Female| 35|                19|                    99|
|        1

In [35]:
#renaming columns for ease of use
df=df.withColumnsRenamed({'Annual Income (k$)':'AnnualIncome','Spending Score (1-100)':'SpendingScore'})

In [36]:
from pyspark.ml.feature import VectorAssembler

feature_assembler = VectorAssembler(inputCols=['Age','AnnualIncome'],outputCol='IndependentFeatures')

In [37]:
output=feature_assembler.transform(df)

In [38]:
output.show()

+----------+------+---+------------+-------------+-------------------+
|CustomerID|Gender|Age|AnnualIncome|SpendingScore|IndependentFeatures|
+----------+------+---+------------+-------------+-------------------+
|         1|  Male| 19|          15|           39|        [19.0,15.0]|
|         2|  Male| 21|          15|           81|        [21.0,15.0]|
|         3|Female| 20|          16|            6|        [20.0,16.0]|
|         4|Female| 23|          16|           77|        [23.0,16.0]|
|         5|Female| 31|          17|           40|        [31.0,17.0]|
|         6|Female| 22|          17|           76|        [22.0,17.0]|
|         7|Female| 35|          18|            6|        [35.0,18.0]|
|         8|Female| 23|          18|           94|        [23.0,18.0]|
|         9|  Male| 64|          19|            3|        [64.0,19.0]|
|        10|Female| 30|          19|           72|        [30.0,19.0]|
|        11|  Male| 67|          19|           14|        [67.0,19.0]|
|     

In [39]:
output.columns

['CustomerID',
 'Gender',
 'Age',
 'AnnualIncome',
 'SpendingScore',
 'IndependentFeatures']

In [40]:
final_df = output.select('SpendingScore', 'IndependentFeatures')

In [41]:
final_df.show()

+-------------+-------------------+
|SpendingScore|IndependentFeatures|
+-------------+-------------------+
|           39|        [19.0,15.0]|
|           81|        [21.0,15.0]|
|            6|        [20.0,16.0]|
|           77|        [23.0,16.0]|
|           40|        [31.0,17.0]|
|           76|        [22.0,17.0]|
|            6|        [35.0,18.0]|
|           94|        [23.0,18.0]|
|            3|        [64.0,19.0]|
|           72|        [30.0,19.0]|
|           14|        [67.0,19.0]|
|           99|        [35.0,19.0]|
|           15|        [58.0,20.0]|
|           77|        [24.0,20.0]|
|           13|        [37.0,20.0]|
|           79|        [22.0,20.0]|
|           35|        [35.0,21.0]|
|           66|        [20.0,21.0]|
|           29|        [52.0,23.0]|
|           98|        [35.0,23.0]|
+-------------+-------------------+
only showing top 20 rows


In [52]:
from pyspark.ml.regression import LinearRegression
train_data,test_data = final_df.randomSplit([0.80,0.20])
model = LinearRegression(featuresCol='IndependentFeatures',labelCol='SpendingScore')
lr = model.fit(train_data)

In [53]:
lr.coefficients

DenseVector([-0.5461, 0.0629])

In [54]:
lr.intercept

67.15616608955906

In [55]:
pred_results = lr.evaluate(test_data)

In [56]:
pred_results.predictions.show()

+-------------+-------------------+------------------+
|SpendingScore|IndependentFeatures|        prediction|
+-------------+-------------------+------------------+
|           11|        [59.0,71.0]| 39.40117328554134|
|           13|        [52.0,88.0]|44.293529904562114|
|           15|        [46.0,98.0]| 48.19939263433123|
|           16|       [47.0,120.0]|49.037271169032195|
|           17|        [43.0,78.0]| 48.57958915098076|
|           23|       [34.0,103.0]|55.067477184928435|
|           31|        [40.0,29.0]| 47.13541324367767|
|           34|        [25.0,72.0]| 58.03243915882967|
|           35|        [65.0,38.0]| 34.04839449539601|
|           40|        [31.0,17.0]| 51.29565370223751|
|           42|        [31.0,40.0]| 52.74256976261373|
|           46|        [19.0,64.0]| 60.80593279786956|
|           47|        [59.0,54.0]| 38.33171358874152|
|           49|        [48.0,60.0]| 44.71657940152229|
|           50|        [27.0,60.0]|56.185270168185156|
|         